# 04 - Data Types, Casting, and Bad Records

**Suggested time: 16 minutes**

Raw extracts frequently represent every value as text. Reliable pipelines convert those values deliberately before using them.

## Learning objectives

By the end of this notebook, you will be able to:

- recognise common Spark data types;
- convert known-valid values with `cast`;
- convert uncertain values safely with `try_cast`; and
- turn failed conversions into a data-quality flag.

## Prerequisite recap

Notebook 03 introduced column expressions and `withColumn`. This lesson applies those tools to raw strings and creates trusted typed columns.

## Common types

- `STRING` stores text.
- `INT` and `LONG` store whole numbers.
- `DOUBLE` stores approximate decimal values.
- `DECIMAL(p,s)` stores fixed-precision values such as currency.
- `BOOLEAN` stores true or false.
- `DATE` stores a calendar date.
- `TIMESTAMP` stores a date and time.

Arrays, maps, and structs are useful for nested data but are outside today's core path.

In [ ]:
from pyspark.sql import functions as F

sales_raw = spark.createDataFrame(
    [
        ('1001', '24.50', '2', 'true', '2026-01-05', '2026-01-05 08:30:00'),
        ('1002', '19.99', '1', 'false', '2026-01-06', '2026-01-06 13:45:00'),
        ('1003', 'not available', 'three', 'unknown', '2026-01-07', 'bad timestamp'),
    ],
    ['order_id_text', 'unit_price_text', 'quantity_text', 'priority_text', 'order_date_text', 'created_at_text'],
)

sales_raw.show(truncate=False)
sales_raw.printSchema()

## SQL expressions with `F.expr`

Notebook 03 introduced `F.col` and `withColumn`. `F.expr('...')` lets you write a column expression in Spark SQL. We use it below because `try_cast` is a SQL expression.

## Cast known-valid values

Use `cast` when the input is controlled and values are known to be valid. Currency is converted to a fixed-precision decimal so cents are represented predictably.

In [ ]:
valid_sales = sales_raw.filter(F.col('order_id_text') != '1003').select(
    F.col('order_id_text').cast('int').alias('order_id'),
    F.col('unit_price_text').cast('decimal(10,2)').alias('unit_price'),
    F.col('quantity_text').cast('int').alias('quantity'),
    F.col('priority_text').cast('boolean').alias('is_priority'),
    F.col('order_date_text').cast('date').alias('order_date'),
    F.col('created_at_text').cast('timestamp').alias('created_at'),
)

valid_sales.show()
valid_sales.printSchema()

## Convert uncertain values safely

A normal cast may fail when strict ANSI behaviour is enabled. SQL `try_cast` returns `NULL` for a value that cannot be converted, allowing the pipeline to identify and route bad records.

In [ ]:
sales_typed = sales_raw.select(
    'order_id_text',
    'unit_price_text',
    'quantity_text',
    'priority_text',
    'created_at_text',
    F.expr('try_cast(order_id_text AS INT)').alias('order_id'),
    F.expr('try_cast(unit_price_text AS DECIMAL(10,2))').alias('unit_price'),
    F.expr('try_cast(quantity_text AS INT)').alias('quantity'),
    F.expr('try_cast(priority_text AS BOOLEAN)').alias('is_priority'),
    F.expr('try_cast(order_date_text AS DATE)').alias('order_date'),
    F.expr('try_cast(created_at_text AS TIMESTAMP)').alias('created_at'),
)

sales_typed.show(truncate=False)
sales_typed.printSchema()

## Flag conversion failures

Keep raw values while validating conversions. A null typed value alongside a non-null raw value is evidence that conversion failed.

In [ ]:
sales_quality = sales_typed.withColumn(
    'needs_review',
    F.col('unit_price').isNull()
    | F.col('quantity').isNull()
    | F.col('is_priority').isNull()
    | F.col('created_at').isNull(),
)

sales_quality.select(
    'order_id_text', 'unit_price_text', 'quantity_text', 'needs_review'
).show()
sales_quality.filter(F.col('needs_review')).show(truncate=False)

## Schema on read or cleanup after read?

Use an explicit typed schema when the source contract is trustworthy. Read questionable fields as strings when you need to preserve malformed values, then use safe conversions and quality flags.

## Your turn

Create `products_typed` from `products_raw`. Convert the ID to an integer, price to `decimal(10,2)`, stock flag to boolean, and update time to timestamp. Add `needs_review` when price conversion fails, and remove the temporary raw-price column from the final result.

In [ ]:
products_raw = spark.createDataFrame(
    [
        ('501', '12.50', 'true', '2026-02-01 10:00:00'),
        ('502', 'price pending', 'false', '2026-02-01 11:30:00'),
    ],
    ['product_id_text', 'price_text', 'in_stock_text', 'last_updated_text'],
)

# Write your solution here.

### Expected result

The result has two rows. Product 501 has price `12.50` and `needs_review = false`. Product 502 has a null price and `needs_review = true`.

### Solution - reveal after attempting

In [ ]:
products_typed = products_raw.select(
    F.col('product_id_text').cast('int').alias('product_id'),
    'price_text',
    F.expr('try_cast(price_text AS DECIMAL(10,2))').alias('price'),
    F.expr('try_cast(in_stock_text AS BOOLEAN)').alias('in_stock'),
    F.expr('try_cast(last_updated_text AS TIMESTAMP)').alias('last_updated'),
).withColumn(
    'needs_review',
    F.col('price').isNull(),
).drop('price_text')

products_typed.show()
products_typed.printSchema()

## Key takeaway

Convert fields deliberately and preserve evidence of bad source values long enough to validate them.

**Next:** handle missing, blank, and duplicate records.